# ENVIRONMENT SETUP AND IMPORTS

In [35]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================

import sys
import os
import subprocess

# --- AUTOMATIC ENVIRONMENT HEALER ---
# Safely handles background installations inside Jupyter without syntax errors
required_packages = {
    "pdfplumber": "pdfplumber",
    "openai": "openai",
    "plotly": "plotly",
    "pandas": "pandas"
}

for module_name, package_name in required_packages.items():
    try:
        __import__(module_name)
    except ImportError:
        print(f"⏳ '{package_name}' not found. Installing package automatically...")
        # Using subprocess prevents the SyntaxError caused by using '!' inside try/except blocks
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "--quiet"])

# ---------------------------------------------------------------------------------
# CORE DEPENDENCY IMPORTS

# 1. UI & Application Layouts
import ipywidgets as widgets
from IPython.display import display, HTML

# 2. In-Memory Operations & File Handling
import io
import pdfplumber

# 3. Database Management & Timestamps
import sqlite3
from datetime import datetime

# 4. Data Processing & Parsing
import json
import pandas as pd

# 5. Interactive Visualizations
import plotly.graph_objects as go

# 6. Generative AI Engine Configuration
from openai import OpenAI

# Initialize the authenticated OpenAI client engine with your explicit key
client = OpenAI(
    api_key="sk-proj-YOUR_ACTUAL_SECRET_OPENAI_KEY_HERE"
)

print("✅ Cell 1 Complete: All full-stack dependencies and API Keys successfully verified and loaded.")

✅ Cell 1 Complete: All full-stack dependencies and API Keys successfully verified and loaded.


# DATABASE

In [36]:
# ==========================================
# CELL 2: DATABASE INITIALIZATION & LOGGER
# ==========================================

def init_local_database():
    """
    Initializes a local SQLite database file and establishes the structured
    relational schema for tracking past analysis trends.
    """
    # Connects to the database file (creates it if it does not exist)
    connection = sqlite3.connect('resume_analyzer.db')
    db_cursor = connection.cursor()
    
    # Create the logging ledger table safely
    db_cursor.execute('''
        CREATE TABLE IF NOT EXISTS processed_resumes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT NOT NULL,
            candidate_name TEXT,
            target_role TEXT,
            match_score INTEGER
        )
    ''')
    
    connection.commit()
    connection.close()
    print("💾 SQLite Database Engine and 'processed_resumes' ledger table successfully initialized.")


def log_analysis_to_db(parsed_json_data, target_role_input="Software Engineer"):
    """
    Commits structured metadata elements extracted from the profile audit
    straight into the persistent local transaction log.
    """
    connection = sqlite3.connect('resume_analyzer.db')
    db_cursor = connection.cursor()
    
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    candidate_name = parsed_json_data.get('candidate_name', 'Unknown Candidate')
    match_score = parsed_json_data.get('match_score', 0)
    
    db_cursor.execute('''
        INSERT INTO processed_resumes (timestamp, candidate_name, target_role, match_score)
        VALUES (?, ?, ?, ?)
    ''', (current_time, candidate_name, target_role_input, match_score))
    
    connection.commit()
    connection.close()
    print(f"✅ Transaction Log Saved: Committed entry for '{candidate_name}' ({match_score}%) to the database ledger.")

# Run the initialization sequence immediately upon cell execution
init_local_database()

💾 SQLite Database Engine and 'processed_resumes' ledger table successfully initialized.


# CORE PIPLINES AND PARSERS

In [37]:
# ==========================================
# CELL 3: CORE AI PIPELINES & PARSERS
# ==========================================

def extract_text_from_pdf(file_bytes_data):
    """
    Parses structural binary data streamed directly from the file upload widget
    and extracts clean string text from all pages.
    """
    extracted_text = ""
    # Treat byte streams seamlessly like an opened file descriptor
    with pdfplumber.open(io.BytesIO(file_bytes_data)) as opened_pdf:
        for page in opened_pdf.pages:
            page_content = page.extract_text()
            if page_content:
                extracted_text += page_content + "\n"
                
    return extracted_text.strip()


def analyze_resume_with_ai(resume_text_content, target_job_description):
    """
    Leverages OpenAI structural parsing models to run an objective ATS audit against 
    the parsed profile, strictly returning a reliable JSON dataset.
    """
    # System engineering instructions specifying precise structured response constraints
    system_instruction = (
        "You are an elite Applicant Tracking System (ATS) optimization engine. "
        "Your task is to analyze the user's resume objectively against the target job description.\n\n"
        "You MUST respond ONLY with a single, valid JSON object matching this schema exactly:\n"
        "{\n"
        "  \"candidate_name\": \"Full Name found on resume or Parsed Candidate\",\n"
        "  \"match_score\": 78,\n"
        "  \"strengths\": [\"Bullet point 1 detailing core technical alignment\", \"Bullet point 2\"],\n"
        "  \"weaknesses\": [\"Bullet point 1 detailing critical qualification or resume structure gaps\", \"Bullet point 2\"],\n"
        "  \"missing_skills\": [\"Specific Skill Keyword 1\", \"Specific Skill Keyword 2\"]\n"
        "}"
    )
    
    user_payload = f"--- TARGET JOB DESCRIPTION ---\n{target_job_description}\n\n--- CANDIDATE RESUME TEXT ---\n{resume_text_content}"
    
    # API extraction call utilizing explicit response formatting options
    api_call_completion = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},  # Hard enforcement of JSON structure at the API level
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_payload}
        ]
    )
    
    # Safe JSON deserialization out of the raw response string block
    parsed_json_output = json.loads(api_call_completion.choices[0].message.content)
    return parsed_json_output

print("⚙️ Cell 3 Complete: Document text extraction and structural AI parsers fully functional.")

⚙️ Cell 3 Complete: Document text extraction and structural AI parsers fully functional.


# COMPONENT FRAMEWORK

In [38]:
# ==========================================
# CELL 4: FRONTEND INPUT COMPONENT FRAMEWORK
# ==========================================

# 1. Instantiate Core Interactive Elements
# Widget component configuration for uploading a single local PDF resume structure
file_upload_widget = widgets.FileUpload(
    accept='.pdf', 
    multiple=False, 
    description="Upload CV (PDF)",
    button_style='info',
    icon='upload'
)

# Multi-line text field configuration for dropping the target job specification metrics
job_desc_widget = widgets.Textarea(
    placeholder="Paste target job description requirements and rules here...", 
    layout=widgets.Layout(height='160px', width='100%')
)

# Launch trigger component to initiate the backend parsing pipelines
process_btn_widget = widgets.Button(
    description="Analyze Profile", 
    button_style='success', 
    icon='rocket',
    layout=widgets.Layout(width='180px', height='40px')
)

# 2. Main Application Asynchronous Output Viewport Mount
app_output_area = widgets.Output()

# 3. Assemble Components Inside a Styled Layout Card Container Panel
input_panel = widgets.VBox([
    widgets.HTML("""
        <div style='margin-bottom: 10px;'>
            <h3 style='color:#1e293b; font-family:sans-serif; margin:0; font-size:18px;'>📝 Step 1: Input Assets</h3>
            <p style='color:#64748b; font-family:sans-serif; font-size:13px; margin:4px 0 0 0;'>
                Provide a target job description and your matching profile to generate an instant audit report.
            </p>
        </div>
    """),
    job_desc_widget,
    widgets.HTML("<div style='margin-top:12px;'></div>"),
    file_upload_widget,
    widgets.HTML("<br>"),
    process_btn_widget
], layout=widgets.Layout(
    padding='24px', 
    border='1px solid #e2e8f0', 
    border_radius='12px', 
    background_color='#f8fafc',
    margin='10px 0'
))

# Render and mount the component canvas locally inside the notebook cell
display(widgets.HTML("<h1 style='color:#0f172a; font-family:sans-serif; border-bottom: 3px solid #e2e8f0; padding-bottom: 12px; margin-bottom:15px;'>AI Full-Stack Resume Analytics Hub</h1>"))
display(input_panel)
display(app_output_area)

HTML(value="<h1 style='color:#0f172a; font-family:sans-serif; border-bottom: 3px solid #e2e8f0; padding-bottom…

Output()

# GENERATOR MODULE

In [39]:
# ==========================================
# CELL 5: AI MOCK INTERVIEW GENERATOR MODULE
# ==========================================

def generate_interview_questions(missing_skills_list, target_role="Specialist"):
    """
    Queries the AI engine to design targeted technical interview preparation 
    questions specifically centered around filling the candidate's parsed skill gaps.
    """
    # Defensive guard to ensure the model has context even if the resume has perfect keyword matches
    if not missing_skills_list:
        missing_skills_list = ["Advanced Systems Architecture & Optimization Concepts"]
        
    system_instruction = (
        f"You are a seasoned principal technical interviewer recruiting for a high-bar {target_role} position. "
        "Review the provided list of skills where the candidate is weak or missing experience. "
        "Generate exactly 3 structured technical interview questions designed to test potential in these specific areas.\n\n"
        "You MUST respond ONLY with a single, valid JSON object matching this schema exactly:\n"
        "{\n"
        "  \"questions\": [\n"
        "    {\n"
        "      \"id\": 1,\n"
        "      \"skill\": \"Name of the specific skill evaluated\",\n"
        "      \"question\": \"The actual open-ended technical scenario or question text.\",\n"
        "      \"hint\": \"A structured tip explaining what a perfect answer should highlight (e.g., concrete metrics, specific design patterns).\"\n"
        "    }\n"
        "  ]\n"
        "}"
    )
    
    user_payload = f"Target Vulnerabilities Matrix: {', '.join(missing_skills_list)}"
    
    # Trigger structural generation call matching our established production patterns
    api_call_completion = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_payload}
        ]
    )
    
    # Safe JSON deserialization out of the raw response string block
    parsed_questions_output = json.loads(api_call_completion.choices[0].message.content)
    return parsed_questions_output

print("🎙️ Cell 5 Complete: AI Mock Interview and question generation engine fully prepared.")

🎙️ Cell 5 Complete: AI Mock Interview and question generation engine fully prepared.


# DASHBOARD

In [40]:
# ==========================================
# CELL 6: APPLICATION ROUTER & DASHBOARD RENDER (FIXED)
# ==========================================

def build_score_gauge_chart(score_value):
    """
    Dynamically generates a modern metrics gauge chart highlighting
    the overall ATS job-match compatibility index.
    """
    gauge_fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score_value,
        title={'text': "ATS Compatibility Index", 'font': {'size': 18, 'color': '#1e293b', 'family': 'sans-serif, Arial'}},
        gauge={
            'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "#475569"},
            'bar': {'color': "#2563eb"}, 
            'bgcolor': "white",
            'borderwidth': 1,
            'bordercolor': "#e2e8f0",
            'steps': [
                {'range': [0, 55], 'color': '#fee2e2'},   
                {'range': [55, 78], 'color': '#fef3c7'},  
                {'range': [78, 100], 'color': '#d1e7dd'}  
            ],
        }
    ))
    
    gauge_fig.update_layout(
        height=200, 
        margin=dict(l=35, r=35, t=50, b=10),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    return gauge_fig


def execute_application_pipeline(button_click_event):
    """
    Asynchronous event router tracking interactions. Extracts asset state, 
    queries the AI layer, updates the SQLite database, and refreshes the dashboard UI.
    """
    with app_output_area:
        # Clear out any stale viewports from previous iterations
        app_output_area.clear_output()
        
        # 1. Frontend State Validation Check
        if not file_upload_widget.value:
            display(HTML("<p style='color:#dc2626; font-family:sans-serif; font-weight:600; margin:10px 0;'>❌ Process Halted: Please upload your resume PDF file first.</p>"))
            return
        if not job_desc_widget.value.strip():
            display(HTML("<p style='color:#dc2626; font-family:sans-serif; font-weight:600; margin:10px 0;'>❌ Process Halted: Please paste the target job description requirements.</p>"))
            return
            
        print("⚡ Stream initiated! Extracting document text matrix...")
        
        try:
            # === FIX: Safe extraction supporting both tuple (ipywidgets v8+) and dict (v7) formats ===
            uploaded_value = file_upload_widget.value
            
            if isinstance(uploaded_value, tuple):
                # New ipywidgets format: it's a tuple of dictionaries
                file_info = uploaded_value[0]
                file_raw_bytes = file_info['content']
                uploaded_file_name = file_info.get('name', 'Uploaded Resume')
            elif isinstance(uploaded_value, dict):
                # Old ipywidgets format: it's a dictionary of dictionaries
                uploaded_file_name = list(uploaded_value.keys())[0]
                file_raw_bytes = uploaded_value[uploaded_file_name]['content']
            else:
                raise TypeError("Unsupported FileUpload widget value format structure.")
            # =====================================================================================
            
            # 3. Process Content Through Backend Modules
            parsed_resume_text = extract_text_from_pdf(file_raw_bytes)
            
            print("🤖 Processing parameters through Generative AI Audit Engine...")
            ai_audit_results = analyze_resume_with_ai(parsed_resume_text, job_desc_widget.value)
            
            # Extract target metrics safely from returned JSON structure
            match_score = ai_audit_results.get('match_score', 0)
            candidate_name = ai_audit_results.get('candidate_name', 'Candidate Profile')
            missing_skills = ai_audit_results.get('missing_skills', [])
            
            # 4. Commit Structured Logging Metrics to Persistent SQLite Database Ledger
            log_analysis_to_db(ai_audit_results, target_role_input="Dashboard Upload")
            
            print("🎙️ Chaining background pipelines to build localized interview prep materials...")
            mock_interview_data = generate_interview_questions(missing_skills, target_role="Target Role")
            
            # Clean viewport message buffers right before showing the final report layout
            app_output_area.clear_output()
            
            # 5. Build HTML Presentational Card Elements
            display(HTML(f"""
                <div style="background: linear-gradient(135deg, #0f172a, #2563eb); padding: 24px; border-radius: 12px; color: white; margin-top:25px; font-family:sans-serif;">
                    <span style="background: rgba(255,255,255,0.2); padding: 4px 10px; border-radius: 4px; font-size: 11px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px;">Audit Analysis Complete</span>
                    <h2 style="margin: 8px 0 0 0; font-size: 24px; font-weight: 700;">Report Card: {candidate_name}</h2>
                </div>
            """))
            
            # Inject interactive Plotly data visualizations live into output stream
            score_gauge = build_score_gauge_chart(match_score)
            score_gauge.show()
            
            # Dynamic generation layout logic for Tab 1 (Executive Summary Strings)
            strengths_list = "".join([f"<li style='margin-bottom:6px;'>{s}</li>" for s in ai_audit_results.get('strengths', [])])
            weaknesses_list = "".join([f"<li style='margin-bottom:6px;'>{w}</li>" for w in ai_audit_results.get('weaknesses', [])])
            
            summary_view_html = f"""
            <div style="padding: 20px; font-family: sans-serif; line-height: 1.6; background-color: white;">
                <h4 style="color: #16a34a; margin-top:0; border-bottom: 1px solid #f1f5f9; padding-bottom:6px;">✅ Structural Alignments & Strengths</h4>
                <ul style="padding-left:20px; color:#334155;">{strengths_list if strengths_list else "<li>General alignment confirmed.</li>"}</ul>
                <h4 style="color: #dc2626; margin-top:20px; border-bottom: 1px solid #f1f5f9; padding-bottom:6px;">⚠️ Qualification Deficiencies & Critiques</h4>
                <ul style="padding-left:20px; color:#334155;">{weaknesses_list if weaknesses_list else "<li>No critical optimization defects caught.</li>"}</ul>
            </div>
            """
            
            # Dynamic generation layout logic for Tab 2 (Keyword Gap Matrix Tags)
            keyword_pill_tags = "".join([
                f'<span style="background:#f0fdf4; color:#16a34a; padding: 6px 12px; border-radius: 6px; font-weight:600; font-size:12px; border: 1px solid #bbf7d0; font-family:sans-serif;">{skill}</span>' 
                for skill in missing_skills
            ])
            
            keyword_view_html = f"""
            <div style="padding: 20px; font-family: sans-serif; background-color: white;">
                <p style='color:#475569; margin-top:0; font-size:14px;'>Inject these parsed missing system constraints organically into your profile document to satisfy high-pass algorithmic parsing filters:</p>
                <div style="display: flex; gap: 10px; flex-wrap: wrap; margin-top: 15px;">
                    {keyword_pill_tags if keyword_pill_tags else "<span style='color:#64748b;'>Zero severe keyword alignment gaps discovered.</span>"}
                </div>
            </div>
            """
            
            # Dynamic generation layout logic for Tab 3 (Interview Question Components)
            interview_cards_list = []
            for item in mock_interview_data.get('questions', []):
                interview_cards_list.append(f"""
                    <div style="border-left: 4px solid #2563eb; background-color: #f8fafc; padding: 16px; margin-bottom: 14px; border-radius: 0 8px 8px 0; font-family:sans-serif;">
                        <span style="background: #dbeafe; color: #1e40af; font-size: 11px; font-weight: bold; padding: 3px 8px; border-radius: 4px;">Evaluated Gap: {item.get('skill')}</span>
                        <p style="margin: 10px 0; font-weight: 600; color: #0f172a; font-size:15px;">Q: {item.get('question')}</p>
                        <p style="margin: 0; font-size: 13px; color: #475569; line-height:1.5; background: white; padding: 10px; border: 1px dashed #e2e8f0; border-radius:6px;">💡 <b>Suggested Hint Architecture:</b> {item.get('hint')}</p>
                    </div>
                """)
                
            interview_view_html = f"""
            <div style="padding: 20px; background-color: white;">
                <h4 style="color: #0f172a; font-family: sans-serif; margin-top:0; margin-bottom:12px; font-size:16px;">Target Technical Screening Drills</h4>
                {"".join(interview_cards_list) if interview_cards_list else "<p style='font-family:sans-serif; color:#64748b;'>No simulated prep drill cards generated.</p>"}
            </div>
            """
            
            # 6. Instantiate and Display the Interactive Multi-Tab Layout Container
            tab_view_panel = widgets.Tab()
            tab_view_panel.children = [
                widgets.HTML(summary_view_html), 
                widgets.HTML(keyword_view_html), 
                widgets.HTML(interview_view_html)
            ]
            tab_view_panel.set_title(0, '📋 Executive Summary')
            tab_view_panel.set_title(1, '🔍 Keyword Gap Analysis')
            tab_view_panel.set_title(2, '🎙️ AI Mock Interviewer')
            
            display(tab_view_panel)
            
        except Exception as runtime_error:
            display(HTML(f"<p style='color:#dc2626; font-family:sans-serif;'>⚠️ Pipeline runtime structural crash: {str(runtime_error)}</p>"))

# Bind the main workflow routing function to the process button click event
process_btn_widget.on_click(execute_application_pipeline)

print("🎯 Cell 6 Complete: Version-safe wrapper compiled successfully.")

🎯 Cell 6 Complete: Version-safe wrapper compiled successfully.


# ADMIN PANEL

In [41]:
# ==========================================
# CELL 7: ADMIN DATABASE UTILITY VIEW PANEL
# ==========================================

def display_historical_logs():
    """
    Queries the persistent local SQLite ledger to display an administrative
    dataframe log of all processed candidates and historical trends.
    """
    try:
        # Establish temporary connection to read tracking metrics
        connection = sqlite3.connect('resume_analyzer.db')
        
        # Pull transactional query directly into a Pandas DataFrame
        query_string = "SELECT id, timestamp, candidate_name, target_role, match_score FROM processed_resumes ORDER BY id DESC"
        historical_df = pd.read_sql_query(query_string, connection)
        
        connection.close()
        
        # Viewport Rendering Logic
        if historical_df.empty:
            display(HTML("""
                <div style='padding: 15px; background-color: #fff8e1; border-left: 4px solid #ffb300; font-family: sans-serif; border-radius:4px;'>
                    <b>📋 System Ledger Empty:</b> No tracking histories saved yet. Run an analysis above to populate this ledger!
                </div>
            """))
        else:
            display(HTML("<h3 style='color:#0f172a; font-family:sans-serif; margin-bottom: 10px;'>📊 Saved Audit Logs Ledger</h3>"))
            # Custom styled dataframe output render
            display(historical_df.style.set_properties(**{
                'background-color': '#ffffff',
                'border-color': '#e2e8f0',
                'color': '#334155',
                'font-family': 'sans-serif'
            }).hide(axis='index'))
            
            # Print quick baseline metric summary strings
            avg_score = historical_df['match_score'].mean()
            print(f"\n📈 Quick Statistics: Total Profiles Audited: {len(historical_df)} | Average ATS Score Match: {avg_score:.1f}%")
            
    except Exception as db_error:
        print(f"❌ Could not access ledger records: {str(db_error)}")

# Call the view window layout function
display_historical_logs()